# Oracle vs Probabilistic Baseline Comparison

Compares the oracle GBR against logistic regression and GBR-classifier baselines on precision, recall, F1, and calibration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import json
import pickle as pkl
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score
)
from sklearn.calibration import calibration_curve

print('Libraries loaded.')

## Load Data — Same 258-Feature Set as Oracle HPO

In [ ]:
def load_unet_result(path):
    df = pd.read_csv(path, index_col='Unnamed: 0')
    df.index = [idx.split('-seg')[0]  # strip '-seg' suffix from U-Net inference output filenames for idx in df.index]
    df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard']  # Dice is the primary metric; Jaccard unused, axis=1, inplace=True)
    return df

def build_feature_df(feature_list, analysis_type, location):
    path = f'../../Results/Analysis_Results/Radiomics/{location}/{analysis_type}.pkl'
    with open(path, 'rb') as f:
        raw = pkl.load(f)
    frames = []
    for feat_name, patient_dict in raw.items():
        feat_df = pd.DataFrame.from_dict(patient_dict, orient='index').astype(float)
        feat_df.columns = [f'{feat_name}_{col}_{analysis_type}' for col in feat_df.columns]
        frames.append(feat_df)
    df = pd.concat(frames, axis=1)
    available = [f for f in feature_list if f in df.columns]
    return df[available]

performance_df = load_unet_result('../../Results/Result/Vanilla_Unet/Unet_test_dice.csv')

# The same 258-feature lists from Oracle Model with HPO
location = 'Tumor_WT'

shape_features = [
    "original_shape_Elongation_flair_shape", "original_shape_Flatness_flair_shape",
    "original_shape_LeastAxisLength_flair_shape", "original_shape_MajorAxisLength_flair_shape",
    "original_shape_MinorAxisLength_flair_shape", "original_shape_Sphericity_flair_shape",
]
size_features = [
    "diagnostics_Mask-original_VolumeNum_flair_size", "original_shape_MeshVolume_flair_size",
    "original_shape_SurfaceArea_flair_size", "original_shape_SurfaceVolumeRatio_flair_size",
]
intensity_features = [
    "diagnostics_Image-original_Mean_flair_intensity", "diagnostics_Image-original_Mean_t2_intensity",
    "diagnostics_Image-original_Mean_t1_intensity",    "diagnostics_Image-original_Mean_t1ce_intensity",
    "diagnostics_Image-original_Maximum_flair_intensity", "diagnostics_Image-original_Maximum_t2_intensity",
    "diagnostics_Image-original_Maximum_t1_intensity",    "diagnostics_Image-original_Maximum_t1ce_intensity",
]
firstorder_features = [
    "original_firstorder_Energy_flair_firstorder",    "original_firstorder_Energy_t2_firstorder",
    "original_firstorder_Energy_t1_firstorder",       "original_firstorder_Energy_t1ce_firstorder",
    "original_firstorder_Entropy_flair_firstorder",   "original_firstorder_Entropy_t2_firstorder",
    "original_firstorder_Entropy_t1_firstorder",      "original_firstorder_Entropy_t1ce_firstorder",
    "original_firstorder_Kurtosis_flair_firstorder",  "original_firstorder_Kurtosis_t2_firstorder",
    "original_firstorder_Kurtosis_t1_firstorder",     "original_firstorder_Kurtosis_t1ce_firstorder",
    "original_firstorder_Skewness_flair_firstorder",  "original_firstorder_Skewness_t2_firstorder",
    "original_firstorder_Skewness_t1_firstorder",     "original_firstorder_Skewness_t1ce_firstorder",
    "original_firstorder_Uniformity_flair_firstorder","original_firstorder_Uniformity_t2_firstorder",
    "original_firstorder_Uniformity_t1_firstorder",   "original_firstorder_Uniformity_t1ce_firstorder",
]
ngtdm_features = [
    "original_ngtdm_Busyness_flair_ngtdm_10",   "original_ngtdm_Busyness_t2_ngtdm_10",
    "original_ngtdm_Busyness_t1_ngtdm_10",      "original_ngtdm_Busyness_t1ce_ngtdm_10",
    "original_ngtdm_Coarseness_flair_ngtdm_10", "original_ngtdm_Coarseness_t2_ngtdm_10",
    "original_ngtdm_Coarseness_t1_ngtdm_10",    "original_ngtdm_Coarseness_t1ce_ngtdm_10",
    "original_ngtdm_Complexity_flair_ngtdm_10", "original_ngtdm_Complexity_t2_ngtdm_10",
    "original_ngtdm_Complexity_t1_ngtdm_10",    "original_ngtdm_Complexity_t1ce_ngtdm_10",
    "original_ngtdm_Contrast_flair_ngtdm_10",   "original_ngtdm_Contrast_t2_ngtdm_10",
    "original_ngtdm_Contrast_t1_ngtdm_10",      "original_ngtdm_Contrast_t1ce_ngtdm_10",
    "original_ngtdm_Strength_flair_ngtdm_10",   "original_ngtdm_Strength_t2_ngtdm_10",
    "original_ngtdm_Strength_t1_ngtdm_10",      "original_ngtdm_Strength_t1ce_ngtdm_10",
]
glcm_features = [
    "original_glcm_Autocorrelation_flair_glcm_10", "original_glcm_Autocorrelation_t2_glcm_10",
    "original_glcm_Autocorrelation_t1_glcm_10",   "original_glcm_Autocorrelation_t1ce_glcm_10",
    "original_glcm_ClusterProminence_flair_glcm_10","original_glcm_ClusterProminence_t2_glcm_10",
    "original_glcm_ClusterProminence_t1_glcm_10",  "original_glcm_ClusterProminence_t1ce_glcm_10",
    "original_glcm_ClusterShade_flair_glcm_10",    "original_glcm_ClusterShade_t2_glcm_10",
    "original_glcm_ClusterShade_t1_glcm_10",       "original_glcm_ClusterShade_t1ce_glcm_10",
    "original_glcm_ClusterTendency_flair_glcm_10", "original_glcm_ClusterTendency_t2_glcm_10",
    "original_glcm_ClusterTendency_t1_glcm_10",    "original_glcm_ClusterTendency_t1ce_glcm_10",
    "original_glcm_Contrast_flair_glcm_10",        "original_glcm_Contrast_t2_glcm_10",
    "original_glcm_Contrast_t1_glcm_10",           "original_glcm_Contrast_t1ce_glcm_10",
    "original_glcm_Correlation_flair_glcm_10",     "original_glcm_Correlation_t2_glcm_10",
    "original_glcm_Correlation_t1_glcm_10",        "original_glcm_Correlation_t1ce_glcm_10",
    "original_glcm_JointAverage_flair_glcm_10",    "original_glcm_JointAverage_t2_glcm_10",
    "original_glcm_JointAverage_t1_glcm_10",       "original_glcm_JointAverage_t1ce_glcm_10",
    "original_glcm_JointEnergy_flair_glcm_10",     "original_glcm_JointEnergy_t2_glcm_10",
    "original_glcm_JointEnergy_t1_glcm_10",        "original_glcm_JointEnergy_t1ce_glcm_10",
    "original_glcm_JointEntropy_flair_glcm_10",    "original_glcm_JointEntropy_t2_glcm_10",
    "original_glcm_JointEntropy_t1_glcm_10",       "original_glcm_JointEntropy_t1ce_glcm_10",
    "original_glcm_MCC_flair_glcm_10",             "original_glcm_MCC_t2_glcm_10",
    "original_glcm_MCC_t1_glcm_10",               "original_glcm_MCC_t1ce_glcm_10",
]
gldm_features = [
    "original_gldm_DependenceNonUniformity_flair_gldm_10", "original_gldm_DependenceNonUniformity_t2_gldm_10",
    "original_gldm_DependenceNonUniformity_t1_gldm_10",   "original_gldm_DependenceNonUniformity_t1ce_gldm_10",
    "original_gldm_GrayLevelNonUniformity_flair_gldm_10", "original_gldm_GrayLevelNonUniformity_t2_gldm_10",
    "original_gldm_GrayLevelNonUniformity_t1_gldm_10",    "original_gldm_GrayLevelNonUniformity_t1ce_gldm_10",
    "original_gldm_GrayLevelVariance_flair_gldm_10",      "original_gldm_GrayLevelVariance_t2_gldm_10",
    "original_gldm_GrayLevelVariance_t1_gldm_10",         "original_gldm_GrayLevelVariance_t1ce_gldm_10",
    "original_gldm_HighGrayLevelEmphasis_flair_gldm_10",  "original_gldm_HighGrayLevelEmphasis_t2_gldm_10",
    "original_gldm_HighGrayLevelEmphasis_t1_gldm_10",     "original_gldm_HighGrayLevelEmphasis_t1ce_gldm_10",
    "original_gldm_LargeDependenceEmphasis_flair_gldm_10","original_gldm_LargeDependenceEmphasis_t2_gldm_10",
    "original_gldm_LargeDependenceEmphasis_t1_gldm_10",   "original_gldm_LargeDependenceEmphasis_t1ce_gldm_10",
    "original_gldm_LowGrayLevelEmphasis_flair_gldm_10",   "original_gldm_LowGrayLevelEmphasis_t2_gldm_10",
    "original_gldm_LowGrayLevelEmphasis_t1_gldm_10",      "original_gldm_LowGrayLevelEmphasis_t1ce_gldm_10",
    "original_gldm_SmallDependenceEmphasis_flair_gldm_10","original_gldm_SmallDependenceEmphasis_t2_gldm_10",
    "original_gldm_SmallDependenceEmphasis_t1_gldm_10",   "original_gldm_SmallDependenceEmphasis_t1ce_gldm_10",
]
glrlm_features = [
    "original_glrlm_GrayLevelNonUniformity_flair_glrlm",        "original_glrlm_GrayLevelNonUniformity_t2_glrlm",
    "original_glrlm_GrayLevelNonUniformity_t1_glrlm",           "original_glrlm_GrayLevelNonUniformity_t1ce_glrlm",
    "original_glrlm_LongRunEmphasis_flair_glrlm",               "original_glrlm_LongRunEmphasis_t2_glrlm",
    "original_glrlm_LongRunEmphasis_t1_glrlm",                  "original_glrlm_LongRunEmphasis_t1ce_glrlm",
    "original_glrlm_LongRunHighGrayLevelEmphasis_flair_glrlm",  "original_glrlm_LongRunHighGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_LongRunHighGrayLevelEmphasis_t1_glrlm",     "original_glrlm_LongRunHighGrayLevelEmphasis_t1ce_glrlm",
    "original_glrlm_LongRunLowGrayLevelEmphasis_flair_glrlm",   "original_glrlm_LongRunLowGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_LongRunLowGrayLevelEmphasis_t1_glrlm",      "original_glrlm_LongRunLowGrayLevelEmphasis_t1ce_glrlm",
    "original_glrlm_LowGrayLevelRunEmphasis_flair_glrlm",       "original_glrlm_LowGrayLevelRunEmphasis_t2_glrlm",
    "original_glrlm_LowGrayLevelRunEmphasis_t1_glrlm",          "original_glrlm_LowGrayLevelRunEmphasis_t1ce_glrlm",
    "original_glrlm_RunLengthNonUniformity_flair_glrlm",        "original_glrlm_RunLengthNonUniformity_t2_glrlm",
    "original_glrlm_RunLengthNonUniformity_t1_glrlm",           "original_glrlm_RunLengthNonUniformity_t1ce_glrlm",
    "original_glrlm_RunPercentage_flair_glrlm",                 "original_glrlm_RunPercentage_t2_glrlm",
    "original_glrlm_RunPercentage_t1_glrlm",                    "original_glrlm_RunPercentage_t1ce_glrlm",
    "original_glrlm_ShortRunEmphasis_flair_glrlm",              "original_glrlm_ShortRunEmphasis_t2_glrlm",
    "original_glrlm_ShortRunEmphasis_t1_glrlm",                 "original_glrlm_ShortRunEmphasis_t1ce_glrlm",
    "original_glrlm_ShortRunHighGrayLevelEmphasis_flair_glrlm", "original_glrlm_ShortRunHighGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_ShortRunHighGrayLevelEmphasis_t1_glrlm",    "original_glrlm_ShortRunHighGrayLevelEmphasis_t1ce_glrlm",
    "original_glrlm_ShortRunLowGrayLevelEmphasis_flair_glrlm",  "original_glrlm_ShortRunLowGrayLevelEmphasis_t2_glrlm",
    "original_glrlm_ShortRunLowGrayLevelEmphasis_t1_glrlm",     "original_glrlm_ShortRunLowGrayLevelEmphasis_t1ce_glrlm",
]
glszm_features = [
    "original_glszm_LargeAreaEmphasis_flair_glszm",     "original_glszm_LargeAreaEmphasis_t2_glszm",
    "original_glszm_LargeAreaEmphasis_t1_glszm",        "original_glszm_LargeAreaEmphasis_t1ce_glszm",
    "original_glszm_SizeZoneNonUniformity_flair_glszm", "original_glszm_SizeZoneNonUniformity_t2_glszm",
    "original_glszm_SizeZoneNonUniformity_t1_glszm",    "original_glszm_SizeZoneNonUniformity_t1ce_glszm",
    "original_glszm_SmallAreaEmphasis_flair_glszm",     "original_glszm_SmallAreaEmphasis_t2_glszm",
    "original_glszm_SmallAreaEmphasis_t1_glszm",        "original_glszm_SmallAreaEmphasis_t1ce_glszm",
    "original_glszm_ZoneEntropy_flair_glszm",           "original_glszm_ZoneEntropy_t2_glszm",
    "original_glszm_ZoneEntropy_t1_glszm",              "original_glszm_ZoneEntropy_t1ce_glszm",
]
volume_features  = ['ED','ET','NCR','WT_volume','TC_volume','ET_volume','TC_WT_ratio','ET_WT_ratio','ET_TC_ratio']
curvature_features = ['mean_gaussian_curvature','std_gaussian_curvature','pos','neg','pos_count','neg_count']

# Build
shape_df      = build_feature_df(shape_features,      'shape',      location)
size_df       = build_feature_df(size_features,        'size',       location)
intensity_df  = build_feature_df(intensity_features,   'intensity',  location)
firstorder_df = build_feature_df(firstorder_features,  'firstorder', location)
ngtdm_df      = build_feature_df(ngtdm_features,       'ngtdm_10',   location)
glcm_df       = build_feature_df(glcm_features,        'glcm_10',    location)
gldm_df       = build_feature_df(gldm_features,        'gldm_10',    location)
glrlm_df      = build_feature_df(glrlm_features,       'glrlm',      location)
glszm_df      = build_feature_df(glszm_features,       'glszm',      location)
volume_df     = pd.read_csv('../../Results/Analysis_Results/volume/GLI-Tumor_volumns.csv',
                             index_col='Unnamed: 0')[volume_features]
prob_df       = pd.read_csv('../../Results/Analysis_Results/probability/Probability_Tumor_boundary.csv',
                             index_col='Unnamed: 0')
curv_df       = pd.read_csv('../../Results/Analysis_Results/curverature/curverature.csv',
                             index_col='Unnamed: 0')[curvature_features]
sal_df        = pd.read_csv('../../Results/Analysis_Results/Saliency/Saliency.csv',
                             index_col='Unnamed: 0')

dfs = [shape_df, size_df, intensity_df, firstorder_df, ngtdm_df,
       glcm_df, gldm_df, glrlm_df, glszm_df,
       volume_df, prob_df, curv_df, sal_df, performance_df]
full_df = dfs[0]
for d in dfs[1:]:
    full_df = full_df.join(d, how='inner')
full_df = full_df.dropna()
print(f'Full matrix: {full_df.shape}')

WT_THRESH, TC_THRESH, ET_THRESH = 0.91, 0.86, 0.85
dice_cols = ['WT dice', 'TC dice', 'ET dice']
feat_cols = [c for c in full_df.columns if c not in dice_cols]
X = full_df[feat_cols]
y_cont = full_df['WT dice']                                    # continuous target for regression
y_bin  = (full_df['WT dice'] < WT_THRESH).astype(int)          # 1 = poor, 0 = good

n_poor = y_bin.sum()
n_good = (y_bin == 0).sum()
print(f'Binary labels — poor: {n_poor}, good: {n_good}, imbalance ratio: {n_good/n_poor:.1f}:1')

## Run All Models — 5-Fold Stratified CV

In [ ]:
SKF = StratifiedKFold(n_splits=5  # stratified to preserve poor/good class ratio  # 5-fold CV — matches oracle evaluation protocol, shuffle=True, random_state=42)

MODELS = {
    'GBR + threshold\n(published Oracle)': {
        'type': 'regression',
        'model': GradientBoostingRegressor(
            n_estimators=150,  # tuned via Bayesian HPO (see Oracle_Model_with_HPO.ipynb), learning_rate=0.10, max_depth=5,
            subsample=0.75, min_samples_split=3, min_samples_leaf=2, random_state=42
        ),
    },
    'Logistic Regression\n(Elastic Net)': {
        'type': 'classifier',
        'model': LogisticRegression(
            penalty='elasticnet', solver='saga', l1_ratio=0.5,
            max_iter=5000, random_state=42, class_weight='balanced'
        ),
    },
    'GBR Classifier': {
        'type': 'classifier',
        'model': GradientBoostingClassifier(
            n_estimators=150,  # tuned via Bayesian HPO (see Oracle_Model_with_HPO.ipynb), learning_rate=0.10, max_depth=5,
            subsample=0.75, random_state=42
        ),
    },
}

cv_results = {name: {'probs': [], 'labels': [], 'preds': [],
                      'f1_poor': [], 'f1_good': [], 'recall_poor': [],
                      'precision_poor': [], 'auc_roc': [], 'auc_pr': []}
              for name in MODELS}

for model_name, cfg in MODELS.items():
    model = cfg['model']
    mtype = cfg['type']

    all_probs, all_labels = [], []

    for train_idx, test_idx in SKF.split(X, y_bin):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr_bin, y_te_bin = y_bin.iloc[train_idx], y_bin.iloc[test_idx]
        y_tr_cont = y_cont.iloc[train_idx]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s  = scaler.transform(X_te)

        if mtype == 'regression':
            model.fit(X_tr_s, y_tr_cont)
            y_pred_cont = model.predict(X_te_s)
            y_pred_bin  = (y_pred_cont < WT_THRESH).astype(int)
            # Use distance from threshold as pseudo-probability (inverted, clipped)
            prob_poor   = np.clip((WT_THRESH - y_pred_cont) / WT_THRESH + 0.5, 0, 1)
        else:
            model.fit(X_tr_s, y_tr_bin)
            y_pred_bin = model.predict(X_te_s)
            prob_poor  = model.predict_proba(X_te_s)[:, 1]  # P(poor)

        all_probs.extend(prob_poor.tolist())
        all_labels.extend(y_te_bin.tolist())
        cv_results[model_name]['preds'].extend(y_pred_bin.tolist())

        cv_results[model_name]['f1_poor'].append(
            f1_score(y_te_bin, y_pred_bin, pos_label=1, zero_division=0))
        cv_results[model_name]['f1_good'].append(
            f1_score(y_te_bin, y_pred_bin, pos_label=0, zero_division=0))
        cv_results[model_name]['recall_poor'].append(
            classification_report(y_te_bin, y_pred_bin, output_dict=True, zero_division=0)['1']['recall'])
        cv_results[model_name]['precision_poor'].append(
            classification_report(y_te_bin, y_pred_bin, output_dict=True, zero_division=0)['1']['precision'])

    cv_results[model_name]['probs']  = all_probs
    cv_results[model_name]['labels'] = all_labels
    if len(set(all_labels)) > 1:
        cv_results[model_name]['auc_roc'] = roc_auc_score(all_labels, all_probs)
        cv_results[model_name]['auc_pr']  = average_precision_score(all_labels, all_probs)
    label = model_name.replace('\n', ' ')
    r = cv_results[model_name]
    print(f'{label}')
    print(f'  F1(poor)={np.mean(r["f1_poor"]):.3f}±{np.std(r["f1_poor"]):.3f}  '
          f'Recall(poor)={np.mean(r["recall_poor"]):.3f}±{np.std(r["recall_poor"]):.3f}  '
          f'AUC-ROC={r["auc_roc"]:.3f}  AUC-PR={r["auc_pr"]:.3f}')

## Figure 1 — ROC and PR Curves

In [ ]:
COLORS_M = {
    'GBR + threshold\n(published Oracle)': 'steelblue',
    'Logistic Regression\n(Elastic Net)':  'darkorange',
    'GBR Classifier':                       'seagreen',
}
STYLES = {
    'GBR + threshold\n(published Oracle)': '-',
    'Logistic Regression\n(Elastic Net)':  '--',
    'GBR Classifier':                       ':',
}
SHORT = {
    'GBR + threshold\n(published Oracle)': 'GBR Regression+Threshold',
    'Logistic Regression\n(Elastic Net)':  'Logistic Regression (Elastic Net)',
    'GBR Classifier':                       'GBR Classifier',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax = axes[0]
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.50)')
for name in MODELS:
    probs  = np.array(cv_results[name]['probs'])
    labels = np.array(cv_results[name]['labels'])
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = cv_results[name]['auc_roc']
    ax.plot(fpr, tpr, color=COLORS_M[name], linestyle=STYLES[name], linewidth=2,
            label=f'{SHORT[name]} (AUC={auc:.3f})')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Poor Segmentation Detection')
ax.legend(fontsize=8); ax.set_aspect('equal')

# PR
ax = axes[1]
baseline = np.array(cv_results[list(MODELS.keys())[0]]['labels']).mean()
ax.axhline(baseline, color='k', linestyle='--', linewidth=1,
           label=f'Random (AP={baseline:.2f})')
for name in MODELS:
    probs  = np.array(cv_results[name]['probs'])
    labels = np.array(cv_results[name]['labels'])
    prec, rec, _ = precision_recall_curve(labels, probs)
    ap = cv_results[name]['auc_pr']
    ax.plot(rec, prec, color=COLORS_M[name], linestyle=STYLES[name], linewidth=2,
            label=f'{SHORT[name]} (AP={ap:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Poor Segmentation Detection')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_probabilistic_roc_pr.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_probabilistic_roc_pr.pdf')

## Figure 2 — F1/Recall/Precision Bar Comparison

In [ ]:
metrics_plot = [
    ('f1_poor',       'F1 — poor class',       'higher is better'),
    ('recall_poor',   'Recall — poor class',    'higher is better'),
    ('precision_poor','Precision — poor class', 'higher is better'),
    ('f1_good',       'F1 — good class',        'higher is better'),
]

model_names = list(MODELS.keys())
short_names = [SHORT[n] for n in model_names]
x = np.arange(len(model_names))
colors = [COLORS_M[n] for n in model_names]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Oracle Model Comparison: Regression + Threshold vs Probabilistic Classifiers', fontsize=11)

for ax, (metric, label, note) in zip(axes, metrics_plot):
    means = [np.mean(cv_results[n][metric]) for n in model_names]
    stds  = [np.std(cv_results[n][metric])  for n in model_names]
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor='white', width=0.5)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.005,
                f'{m:.3f}', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=15, ha='right', fontsize=7)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel(label)
    ax.set_title(f'{label}\n({note})', fontsize=9)

plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_probabilistic_comparison.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_probabilistic_comparison.pdf')

## Figure 3 — Probability Calibration Curves
Well-calibrated classifiers have curves close to the diagonal.  
The GBR regression pseudo-probabilities may be poorly calibrated compared to logistic regression.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Perfectly calibrated')

for name in MODELS:
    probs  = np.array(cv_results[name]['probs'])
    labels = np.array(cv_results[name]['labels'])
    try:
        frac_pos, mean_pred = calibration_curve(  # reliability diagram: perfect calibration → diagonallabels, probs, n_bins=10, strategy='uniform')
        ax.plot(mean_pred, frac_pos, marker='o', color=COLORS_M[name],
                linestyle=STYLES[name], linewidth=2, label=SHORT[name])
    except Exception as e:
        print(f'Calibration skipped for {name}: {e}')

ax.set_xlabel('Mean predicted probability (P(poor))')
ax.set_ylabel('Fraction of positives (actual poor)')
ax.set_title('Probability Calibration — P(poor segmentation)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_probabilistic_calibration.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_probabilistic_calibration.pdf')

## Figure 4 — Clinical Utility: Detection Rate vs False-Alarm Rate
For each model, shows how many poor cases are caught as the false-alarm budget increases.
Useful for justifying the choice of operating point.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name in MODELS:
    probs  = np.array(cv_results[name]['probs'])
    labels = np.array(cv_results[name]['labels'])
    fpr, tpr, thresholds = roc_curve(labels, probs)
    n_poor_total = labels.sum()
    n_total = len(labels)
    # x-axis: number of false alarms (FP), y-axis: number of poor cases detected (TP)
    n_fp = fpr * (n_total - n_poor_total)
    n_tp = tpr * n_poor_total
    ax.plot(n_fp, n_tp, color=COLORS_M[name], linestyle=STYLES[name],
            linewidth=2, label=SHORT[name])
    # Mark the operating point (published threshold)
    preds = np.array(cv_results[name]['preds'])
    tp_op = ((preds == 1) & (labels == 1)).sum()
    fp_op = ((preds == 1) & (labels == 0)).sum()
    ax.scatter([fp_op], [tp_op], color=COLORS_M[name], zorder=5, s=80, marker='*')

ax.set_xlabel('Number of False Alarms (good cases flagged as poor)')
ax.set_ylabel('Number of Poor Cases Detected')
ax.set_title('Clinical Utility Curve\n(* = operating point at reported threshold)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../../Results/Figures/oracle_probabilistic_utility.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../../Results/Figures/oracle_probabilistic_utility.pdf')

## Summary Table and JSON

In [ ]:
print('=== COMPARISON SUMMARY ===')
SEP = '-' * 90
print(f'{"Model":35s}  {"F1(poor)":>10}  {"Recall(poor)":>13}  {"Prec(poor)":>11}  '
      f'{"F1(good)":>9}  {"AUC-ROC":>8}  {"AUC-PR":>7}')
print(SEP)

for name in MODELS:
    r = cv_results[name]
    short = SHORT[name]
    print(f'{short:35s}  '
          f'{np.mean(r["f1_poor"]):.3f}±{np.std(r["f1_poor"]):.3f}  '
          f'{np.mean(r["recall_poor"]):.3f}±{np.std(r["recall_poor"]):.3f}  '
          f'{np.mean(r["precision_poor"]):.3f}±{np.std(r["precision_poor"]):.3f}  '
          f'{np.mean(r["f1_good"]):.3f}±{np.std(r["f1_good"]):.3f}  '
          f'{r["auc_roc"]:.3f}  '
          f'{r["auc_pr"]:.3f}')

print()
print('Key insight:')
reg_name  = 'GBR + threshold\n(published Oracle)'
logr_name = 'Logistic Regression\n(Elastic Net)'
r_reg  = cv_results[reg_name]
r_logr = cv_results[logr_name]
diff_f1  = np.mean(r_logr['f1_poor'])  - np.mean(r_reg['f1_poor'])
diff_auc = r_logr['auc_roc'] - r_reg['auc_roc']
print(f'  Logistic vs GBR+threshold: Delta-F1(poor)={diff_f1:+.3f}, Delta-AUC-ROC={diff_auc:+.3f}')
if abs(diff_f1) < 0.02:
    print('  --> Negligible difference: both approaches perform similarly.')
    print('      The regression approach is preferred as it provides richer clinical information')
    print('      (continuous Dice score) and allows flexible threshold adjustment.')
elif diff_f1 > 0.02:
    print('  --> Logistic regression outperforms regression+threshold for classification.')
    print('      Consider reporting both or switching to the probabilistic model.')
else:
    print('  --> GBR+threshold outperforms logistic regression.')
    print('      The regression approach is justified.')

In [ ]:
output = {
    'n_features': X.shape[1],
    'n_samples': X.shape[0],
    'n_poor': int(y_bin.sum()),
    'n_good': int((y_bin == 0).sum()),
    'wt_threshold': WT_THRESH,
    'cv': '5-fold StratifiedKFold (random_state=42)',
    'results': {
        SHORT[name]: {
            'f1_poor_mean':        round(float(np.mean(cv_results[name]['f1_poor'])), 4),
            'f1_poor_std':         round(float(np.std(cv_results[name]['f1_poor'])),  4),
            'recall_poor_mean':    round(float(np.mean(cv_results[name]['recall_poor'])), 4),
            'recall_poor_std':     round(float(np.std(cv_results[name]['recall_poor'])),  4),
            'precision_poor_mean': round(float(np.mean(cv_results[name]['precision_poor'])), 4),
            'precision_poor_std':  round(float(np.std(cv_results[name]['precision_poor'])),  4),
            'f1_good_mean':        round(float(np.mean(cv_results[name]['f1_good'])), 4),
            'f1_good_std':         round(float(np.std(cv_results[name]['f1_good'])),  4),
            'auc_roc':             round(float(cv_results[name]['auc_roc']), 4),
            'auc_pr':              round(float(cv_results[name]['auc_pr']),  4),
        }
        for name in MODELS
    },
}
with open('../../Results/Json_summary/oracle_probabilistic_results.json', 'w') as f:
    json.dump(output, f, indent=4)
print('Saved: ../../Results/Json_summary/oracle_probabilistic_results.json')